In [12]:

import os
import joblib
import numpy as np
import pandas as pd

from IPython.display import display, HTML

# ==========================================
# KONFIGURASI DIREKTORI
# ==========================================

BASE_DIR = r"C:\Users\nofal\Downloads\lampiran\arimax_garch"

DATA_DIR = os.path.join(BASE_DIR, "data")
MODEL_DIR = os.path.join(BASE_DIR, "outputs", "model")

# ==========================================
# KONFIGURASI SEGMENTASI
# ==========================================

SEQUENCE_LENGTH = 60

JUMLAH_TRAIN = 914
JUMLAH_TRAIN_HYBRID = 912
JUMLAH_VALIDASI = 244
JUMLAH_TESTING = 305

# ==========================================
# KONFIGURASI PENELITIAN
# ==========================================

KONFIGURASI = [
    ("ANTM", False, "Standalone"),
    ("ANTM", True, "Hybrid"),
    ("INCO", False, "Standalone"),
    ("INCO", True, "Hybrid")
]


In [13]:

def muat_data(EMITEN, USE_GARCH):

    # Membaca dataset utama
    data = pd.read_csv(
        os.path.join(
            DATA_DIR,
            f"data_{EMITEN.lower()}.csv"
        )
    )

    data["Tanggal"] = pd.to_datetime(data["Tanggal"])

    # Menggabungkan volatilitas GARCH
    if USE_GARCH:

        data = data.iloc[2:].reset_index(drop=True)

        vol = pd.read_excel(
            os.path.join(
                BASE_DIR,
                "outputs",
                f"01_arimax_garch_{EMITEN}.xlsx"
            ),
            sheet_name="conditional_volatility"
        )

        vol = vol[["Tanggal", "sigma"]].rename(
            columns={"sigma": "garch_vol"}
        )

        vol["Tanggal"] = pd.to_datetime(vol["Tanggal"])

        data = data.merge(
            vol,
            on="Tanggal",
            how="left"
        )

    # Pemeriksaan data
    jumlah_data = 1521 if USE_GARCH else 1523

    assert len(data) == jumlah_data
    assert data.isna().sum().sum() == 0

    # Menentukan fitur dan target
    FITUR = list(data.columns[1:])
    TARGET = FITUR[0]

    return data, FITUR, TARGET


In [14]:

def muat_scaler(EMITEN, USE_GARCH):

    nama_model = "GARCH-LSTM" if USE_GARCH else "LSTM"

    path_scaler_fitur = os.path.join(
        MODEL_DIR,
        f"scaler_fitur_{nama_model}_{EMITEN}.pkl"
    )

    path_scaler_target = os.path.join(
        MODEL_DIR,
        f"scaler_target_{nama_model}_{EMITEN}.pkl"
    )

    scaler_fitur = joblib.load(path_scaler_fitur)
    scaler_target = joblib.load(path_scaler_target)

    return scaler_fitur, scaler_target


In [15]:

def segmentasi_data(X, y, USE_GARCH):

    n_train = (
        JUMLAH_TRAIN_HYBRID
        if USE_GARCH
        else JUMLAH_TRAIN
    )

    n_val = JUMLAH_VALIDASI
    n_test = JUMLAH_TESTING

    # Batas observasi
    train_end = SEQUENCE_LENGTH + n_train
    val_end = train_end + n_val
    test_end = val_end + n_test

    assert len(X) == test_end

    # Fungsi pembentukan sequence
    def buat_sequence(start, end):

        X_result = []
        y_result = []

        for i in range(start, end):

            X_result.append(
                X[i:i + SEQUENCE_LENGTH]
            )

            y_result.append(
                y[i + SEQUENCE_LENGTH]
            )

        return (
            np.asarray(X_result),
            np.asarray(y_result)
        )

    # Training
    X_train, y_train = buat_sequence(
        0,
        n_train
    )

    # Validation
    X_val, y_val = buat_sequence(
        train_end - SEQUENCE_LENGTH,
        train_end - SEQUENCE_LENGTH + n_val
    )

    # Testing
    X_test, y_test = buat_sequence(
        val_end - SEQUENCE_LENGTH,
        val_end - SEQUENCE_LENGTH + n_test
    )

    return (
        X_train, y_train,
        X_val, y_val,
        X_test, y_test
    )


In [16]:

# ==========================================
# FUNGSI TABEL SEGMENTASI RINGKAS
# ==========================================

def format_angka(nilai):
    return f"{float(nilai):.3f}".replace(".", ",")


def buat_tabel(
    X,
    y,
    FITUR,
    EMITEN,
    SKENARIO,
    JENIS,
    NOMOR_TABEL
):

    jumlah_sampel = len(X)

    # Sampel pertama, kedua, tengah, terakhir
    indeks = [
        0,
        1,
        jumlah_sampel // 2 - 1,
        jumlah_sampel - 1
    ]

    baris = []

    for idx in indeks:

        sequence = X[idx]

        # Format setiap periode sesuai jumlah fitur
        def format_periode(nilai):

            return "(" + "; ".join(
                format_angka(v) for v in nilai
            ) + ")"

        # Dua periode awal dan dua periode akhir
        sequence_ringkas = (
            format_periode(sequence[0])
            + "; "
            + format_periode(sequence[1])
            + "; …; "
            + format_periode(sequence[-2])
            + "; "
            + format_periode(sequence[-1])
        )

        input_text = f"[{sequence_ringkas}]"

        target_text = format_angka(y[idx])

        baris.append({
            "Input Sequence (X)": input_text,
            "Target Value (y)": target_text
        })

    # Menambahkan baris elipsis
    baris.insert(2, {
        "Input Sequence (X)": "⋮",
        "Target Value (y)": "⋮"
    })

    tabel = pd.DataFrame(baris)

    # Judul tabel
    judul = (
        f"Tabel {NOMOR_TABEL} Hasil Segmentasi Data "
        f"{JENIS} {EMITEN} {SKENARIO}"
    )

    display(HTML(
        f"""
        <h4 style="text-align:left; margin-bottom:8px;">
        {judul}
        </h4>
        """
    ))

    # Tampilan tabel sederhana untuk Word
    display(HTML(
        tabel.to_html(
            index=False,
            escape=False,
            border=1,
            justify="center"
        )
        +
        """
        <style>
        table {
            width: 100%;
            border-collapse: collapse;
            font-family: 'Times New Roman';
            font-size: 12pt;
        }
        th, td {
            border: 1px solid black;
            padding: 8px;
            text-align: center;
            vertical-align: middle;
        }
        th {
            font-weight: bold;
        }
        </style>
        """
    ))


In [17]:

# ==========================================
# MENAMPILKAN SELURUH TABEL SEGMENTASI
# ==========================================

HASIL_SEGMENTASI = {}

nomor_tabel = 12

for EMITEN, USE_GARCH, SKENARIO in KONFIGURASI:

    # Membaca dataset
    data, FITUR, TARGET = muat_data(
        EMITEN,
        USE_GARCH
    )

    # Memuat scaler
    scaler_fitur, scaler_target = muat_scaler(
        EMITEN,
        USE_GARCH
    )

    # Normalisasi fitur dan target
    X_scaled = scaler_fitur.transform(
        data[FITUR].to_numpy()
    )

    y_scaled = scaler_target.transform(
        data[[TARGET]].to_numpy()
    ).reshape(-1)

    # Segmentasi
    hasil = segmentasi_data(
        X_scaled,
        y_scaled,
        USE_GARCH
    )

    X_train, y_train, X_val, y_val, X_test, y_test = hasil

    HASIL_SEGMENTASI[(EMITEN, SKENARIO)] = hasil

    # Training
    buat_tabel(
        X_train, y_train,
        FITUR, EMITEN, SKENARIO,
        "Train", nomor_tabel
    )
    nomor_tabel += 1

    # Validation
    buat_tabel(
        X_val, y_val,
        FITUR, EMITEN, SKENARIO,
        "Validation", nomor_tabel
    )
    nomor_tabel += 1

    # Testing
    buat_tabel(
        X_test, y_test,
        FITUR, EMITEN, SKENARIO,
        "Test", nomor_tabel
    )
    nomor_tabel += 1


Input Sequence (X),Target Value (y)
"[(0,173; 0,083; 0,066); (0,177; 0,123; 0,057); …; (0,006; 0,302; 0,007); (0,024; 0,286; 0,007)]","0,039"
"[(0,177; 0,123; 0,057); (0,189; 0,150; 0,058); …; (0,024; 0,286; 0,007); (0,039; 0,243; 0,009)]","0,030"
⋮,⋮
"[(0,715; 0,621; 0,168); (0,704; 0,648; 0,166); …; (0,507; 0,578; 0,232); (0,504; 0,592; 0,244)]","0,539"
"[(0,490; 0,575; 0,152); (0,477; 0,570; 0,147); …; (0,488; 1,000; 0,118); (0,481; 1,000; 0,113)]","0,477"


Input Sequence (X),Target Value (y)
"[(0,477; 0,570; 0,147); (0,481; 0,570; 0,150); …; (0,481; 1,000; 0,113); (0,477; 1,000; 0,113)]","0,488"
"[(0,481; 0,570; 0,150); (0,514; 0,624; 0,158); …; (0,477; 1,000; 0,113); (0,488; 0,983; 0,111)]","0,476"
⋮,⋮
"[(0,462; 1,352; 0,132); (0,465; 1,352; 0,134); …; (0,356; 1,559; 0,117); (0,353; 1,570; 0,114)]","0,349"
"[(0,428; 1,898; 0,137); (0,451; 1,942; 0,133); …; (0,381; 1,972; 0,089); (0,377; 2,003; 0,090)]","0,393"


Input Sequence (X),Target Value (y)
"[(0,451; 1,942; 0,133); (0,441; 1,964; 0,128); …; (0,377; 2,003; 0,090); (0,393; 2,003; 0,093)]","0,398"
"[(0,441; 1,964; 0,128); (0,430; 1,984; 0,127); …; (0,393; 2,003; 0,093); (0,398; 1,983; 0,098)]","0,412"
⋮,⋮
"[(1,039; 3,178; 0,084); (1,032; 3,202; 0,082); …; (1,148; 3,632; 0,085); (1,165; 3,640; 0,083)]","1,113"
"[(1,552; 5,985; 0,151); (1,500; 5,983; 0,142); …; (1,299; 5,142; 0,171); (1,243; 5,092; 0,163)]","1,194"


Input Sequence (X),Target Value (y)
"[(0,189; 0,150; 0,058; 0,229); (0,185; 0,159; 0,060; 0,243); …; (0,039; 0,243; 0,009; 0,640); (0,030; 0,273; 0,009; 0,744)]","0,036"
"[(0,185; 0,159; 0,060; 0,243); (0,196; 0,136; 0,062; 0,232); …; (0,030; 0,273; 0,009; 0,744); (0,036; 0,196; 0,012; 0,728)]","0,031"
⋮,⋮
"[(0,704; 0,648; 0,166; 0,072); (0,697; 0,633; 0,172; 0,068); …; (0,504; 0,592; 0,244; 0,293); (0,539; 0,594; 0,246; 0,273)]","0,527"
"[(0,490; 0,575; 0,152; 0,028); (0,477; 0,570; 0,147; 0,031); …; (0,488; 1,000; 0,118; 0,101); (0,481; 1,000; 0,113; 0,152)]","0,477"


Input Sequence (X),Target Value (y)
"[(0,477; 0,570; 0,147; 0,031); (0,481; 0,570; 0,150; 0,041); …; (0,481; 1,000; 0,113; 0,152); (0,477; 1,000; 0,113; 0,153)]","0,488"
"[(0,481; 0,570; 0,150; 0,041); (0,514; 0,624; 0,158; 0,039); …; (0,477; 1,000; 0,113; 0,153); (0,488; 0,983; 0,111; 0,141)]","0,476"
⋮,⋮
"[(0,462; 1,352; 0,132; 0,134); (0,465; 1,352; 0,134; 0,123); …; (0,356; 1,559; 0,117; 0,147); (0,353; 1,570; 0,114; 0,137)]","0,349"
"[(0,428; 1,898; 0,137; 0,073); (0,451; 1,942; 0,133; 0,076); …; (0,381; 1,972; 0,089; 0,260); (0,377; 2,003; 0,090; 0,244)]","0,393"


Input Sequence (X),Target Value (y)
"[(0,451; 1,942; 0,133; 0,076); (0,441; 1,964; 0,128; 0,123); …; (0,377; 2,003; 0,090; 0,244); (0,393; 2,003; 0,093; 0,227)]","0,398"
"[(0,441; 1,964; 0,128; 0,123); (0,430; 1,984; 0,127; 0,128); …; (0,393; 2,003; 0,093; 0,227); (0,398; 1,983; 0,098; 0,234)]","0,412"
⋮,⋮
"[(1,039; 3,178; 0,084; 0,429); (1,032; 3,202; 0,082; 0,422); …; (1,148; 3,632; 0,085; 0,313); (1,165; 3,640; 0,083; 0,353)]","1,113"
"[(1,552; 5,985; 0,151; 0,347); (1,500; 5,983; 0,142; 0,461); …; (1,299; 5,142; 0,171; 0,229); (1,243; 5,092; 0,163; 0,216)]","1,194"


... (2 outputs left out)

Input Sequence (X),Target Value (y)
"[(0,383; 0,133); (0,377; 0,128); …; (0,272; 0,090); (0,275; 0,093)]","0,275"
"[(0,377; 0,128); (0,372; 0,127); …; (0,275; 0,093); (0,275; 0,098)]","0,310"
⋮,⋮
"[(0,275; 0,084); (0,303; 0,082); …; (0,340; 0,085); (0,347; 0,083)]","0,336"
"[(0,744; 0,151); (0,741; 0,142); …; (0,741; 0,171); (0,744; 0,163)]","0,748"


Input Sequence (X),Target Value (y)
"[(0,289; 0,058; 0,191); (0,293; 0,060; 0,179); …; (0,077; 0,009; 1,000); (0,080; 0,009; 0,964)]","0,100"
"[(0,293; 0,060; 0,179); (0,290; 0,062; 0,165); …; (0,080; 0,009; 0,964); (0,100; 0,012; 0,907)]","0,082"
⋮,⋮
"[(0,460; 0,166; 0,009); (0,453; 0,172; 0,009); …; (0,445; 0,244; 0,107); (0,464; 0,246; 0,097)]","0,462"
"[(0,551; 0,152; 0,013); (0,540; 0,147; 0,041); …; (0,401; 0,118; 0,185); (0,405; 0,113; 0,178)]","0,397"


Input Sequence (X),Target Value (y)
"[(0,540; 0,147; 0,041); (0,565; 0,150; 0,041); …; (0,405; 0,113; 0,178); (0,397; 0,113; 0,162)]","0,412"
"[(0,565; 0,150; 0,041); (0,585; 0,158; 0,083); …; (0,397; 0,113; 0,162); (0,412; 0,111; 0,152)]","0,412"
⋮,⋮
"[(0,369; 0,132; 0,152); (0,368; 0,134; 0,140); …; (0,319; 0,117; 0,196); (0,311; 0,114; 0,182)]","0,326"
"[(0,376; 0,137; 0,179); (0,383; 0,133; 0,163); …; (0,274; 0,089; 0,103); (0,272; 0,090; 0,101)]","0,275"


Input Sequence (X),Target Value (y)
"[(0,383; 0,133; 0,163); (0,377; 0,128; 0,151); …; (0,272; 0,090; 0,101); (0,275; 0,093; 0,090)]","0,275"
"[(0,377; 0,128; 0,151); (0,372; 0,127; 0,139); …; (0,275; 0,093; 0,090); (0,275; 0,098; 0,081)]","0,310"
⋮,⋮
"[(0,275; 0,084; 0,426); (0,303; 0,082; 0,398); …; (0,340; 0,085; 0,287); (0,347; 0,083; 0,265)]","0,336"
"[(0,744; 0,151; 0,405); (0,741; 0,142; 0,377); …; (0,741; 0,171; 0,293); (0,744; 0,163; 0,272)]","0,748"
